In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tqdm.auto import tqdm

tf.random.set_seed(42)
np.random.seed(42)
print("GPU:", tf.config.list_physical_devices("GPU"))   

IMG_SIZE    = 64
BATCH_SIZE  = 128
LATENT_DIM  = 100
EPOCHS      = 25          
NUM_IMAGES  = 60000       

RESULTS_PATH = "/kaggle/working/DCGAN_Faces"
SAMPLES_PATH = os.path.join(RESULTS_PATH, "samples")
os.makedirs(SAMPLES_PATH, exist_ok=True)


files = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True))
print("Total images found:", len(files))
assert len(files) > 0

files = np.random.permutation(files)[:NUM_IMAGES].tolist()

def load_and_preprocess(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.crop_to_bounding_box(img, 40, 15, 148, 148) 
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])               
    img = tf.cast(img, tf.float32)
    img = (img - 127.5) / 127.5                                   
    return img

train_dataset = (
    tf.data.Dataset.from_tensor_slices(files)
    .shuffle(len(files))
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

real_batch = next(iter(train_dataset))
print("Batch shape:", real_batch.shape, "| min/max:",
      float(tf.reduce_min(real_batch)), float(tf.reduce_max(real_batch)))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for img, ax in zip(real_batch[:16], axes.flat):
    ax.imshow((img.numpy() + 1) / 2)
    ax.axis("off")
plt.suptitle("Real images after preprocessing")
plt.show()


init = tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02)

def build_generator():
    return tf.keras.Sequential([
        layers.Input(shape=(LATENT_DIM,)),
        layers.Dense(4 * 4 * 512, use_bias=False, kernel_initializer=init),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Reshape((4, 4, 512)),

        layers.Conv2DTranspose(256, 4, 2, "same", use_bias=False, kernel_initializer=init),  
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2DTranspose(128, 4, 2, "same", use_bias=False, kernel_initializer=init),  
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2DTranspose(64, 4, 2, "same", use_bias=False, kernel_initializer=init),   
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2DTranspose(3, 4, 2, "same", activation="tanh", kernel_initializer=init), 
    ], name="generator")

def build_discriminator():
    return tf.keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        layers.Conv2D(64, 4, 2, "same", kernel_initializer=init),                            
        layers.LeakyReLU(0.2),

        layers.Conv2D(128, 4, 2, "same", use_bias=False, kernel_initializer=init),           
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2D(256, 4, 2, "same", use_bias=False, kernel_initializer=init),           
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2D(512, 4, 2, "same", use_bias=False, kernel_initializer=init),           
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Flatten(),
        layers.Dropout(0.3),
        layers.Dense(1),   
    ], name="discriminator")

generator = build_generator()
discriminator = build_discriminator()
generator.summary()
discriminator.summary()


bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = bce(tf.ones_like(real_output) * 0.9, real_output)   # label smoothing
    fake_loss = bce(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return bce(tf.ones_like(fake_output), fake_output)

generator_optimizer     = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

fixed_noise = tf.random.normal([16, LATENT_DIM], seed=42)   

@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], LATENT_DIM])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(fake_images, training=True)
        gen_loss  = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
    gen_grads  = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    generator_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))
    return gen_loss, disc_loss


def save_epoch_images(epoch, show=False):
    imgs = generator(fixed_noise, training=False).numpy()
    imgs = ((imgs + 1) / 2).clip(0, 1)
    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    for img, ax in zip(imgs, axes.flat):
        ax.imshow(img)
        ax.axis("off")
    plt.suptitle(f"Epoch {epoch}")
    plt.tight_layout()
    plt.savefig(os.path.join(SAMPLES_PATH, f"epoch_{epoch:03d}.png"), dpi=100)
    if show:
        plt.show()
    plt.close()

history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    g_losses, d_losses = [], []

    for batch in tqdm(train_dataset, desc=f"Epoch {epoch}/{EPOCHS}"):
        g, d = train_step(batch)
        g_losses.append(float(g))
        d_losses.append(float(d))

    history.append({
        "Epoch": epoch,
        "Generator Loss": float(np.mean(g_losses)),
        "Discriminator Loss": float(np.mean(d_losses)),
    })
    print(f"Epoch {epoch} | G: {history[-1]['Generator Loss']:.3f} "
          f"| D: {history[-1]['Discriminator Loss']:.3f} "
          f"| {(time.time() - t0) / 60:.1f} min")

    save_epoch_images(epoch, show=(epoch == 1 or epoch % 5 == 0))

    if epoch % 5 == 0:
        generator.save(os.path.join(RESULTS_PATH, "generator_latest.keras"))
        discriminator.save(os.path.join(RESULTS_PATH, "discriminator_latest.keras"))

generator.save(os.path.join(RESULTS_PATH, "generator_final.keras"))
discriminator.save(os.path.join(RESULTS_PATH, "discriminator_final.keras"))
print("Training finished!")


df = pd.DataFrame(history)
df.to_csv(os.path.join(RESULTS_PATH, "history.csv"), index=False)

plt.figure(figsize=(9, 4))
plt.plot(df["Epoch"], df["Generator Loss"], label="Generator")
plt.plot(df["Epoch"], df["Discriminator Loss"], label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Losses")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(RESULTS_PATH, "losses.png"), dpi=150)
plt.show()


sample_files = sorted(glob.glob(os.path.join(SAMPLES_PATH, "epoch_*.png")))
picks = [sample_files[0], sample_files[len(sample_files) // 2], sample_files[-1]]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, f in zip(axes, picks):
    ax.imshow(mpimg.imread(f))
    ax.axis("off")
plt.suptitle("Generated faces over time: Start vs Middle vs End", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, "progress_comparison.png"), dpi=120)
plt.show()


noise = tf.random.normal([25, LATENT_DIM])
faces = generator(noise, training=False).numpy()
faces = ((faces + 1) / 2).clip(0, 1)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for img, ax in zip(faces, axes.flat):
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Final Generated Faces", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, "final_faces.png"), dpi=150)
plt.show()


from PIL import Image
import shutil

frames = [Image.open(f) for f in sample_files]
frames[0].save(os.path.join(RESULTS_PATH, "training_progress.gif"),
               save_all=True, append_images=frames[1:], duration=400, loop=0)

shutil.make_archive("/kaggle/working/DCGAN_results", "zip", RESULTS_PATH)
print("Saved: /kaggle/working/DCGAN_results.zip")


In [ ]:
import os, glob, time, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

OLD_PATH    = "/kaggle/working/DCGAN_Faces"       
NEW_PATH    = "/kaggle/working/DCGAN_Faces_v2"     
LATENT_DIM  = 100
IMG_SIZE    = 64
BATCH_SIZE  = 128
NUM_IMAGES  = 100000
EPOCHS      = 20
LR_START    = 1e-4
LR_END      = 2e-5
EMA_DECAY   = 0.999
NOISE_SCALE = 0.7

SAMPLES_PATH = os.path.join(NEW_PATH, "samples")
os.makedirs(SAMPLES_PATH, exist_ok=True)
print("GPU:", tf.config.list_physical_devices("GPU"))

try:
    generator, discriminator
except NameError:
    generator = tf.keras.models.load_model(os.path.join(OLD_PATH, "generator_final.keras"))
    discriminator = tf.keras.models.load_model(os.path.join(OLD_PATH, "discriminator_final.keras"))

old_generator = tf.keras.models.clone_model(generator)
old_generator.set_weights(generator.get_weights())

ema_generator = tf.keras.models.clone_model(generator)
ema_generator.set_weights(generator.get_weights())

files = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True))
assert len(files) > 0,  (Add Data)"
files = np.random.permutation(files)[:NUM_IMAGES].tolist()
print("Images used:", len(files))

def load_and_preprocess(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.crop_to_bounding_box(img, 40, 15, 148, 148)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return (img - 127.5) / 127.5

train_dataset = (
    tf.data.Dataset.from_tensor_slices(files)
    .shuffle(len(files))
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = bce(tf.ones_like(real_output) * 0.9, real_output)   # label smoothing
    fake_loss = bce(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return bce(tf.ones_like(fake_output), fake_output)

g_opt = tf.keras.optimizers.Adam(LR_START, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(LR_START, beta_1=0.5)

def set_lr(opt, value):
    try:
        opt.learning_rate.assign(value)
    except Exception:
        opt.learning_rate = value

@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], LATENT_DIM])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(fake_images, training=True)
        gen_loss  = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
    g_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    d_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))
    d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    # تحديث الـ EMA
    for e, w in zip(ema_generator.variables, generator.variables):
        e.assign(EMA_DECAY * e + (1.0 - EMA_DECAY) * w)
    return gen_loss, disc_loss

fixed_noise = tf.random.normal([16, LATENT_DIM], seed=7)

def to_images(model, noise):
    imgs = model(noise, training=False).numpy()
    return ((imgs + 1) / 2).clip(0, 1)

def save_epoch_images(epoch, show=False):
    imgs = to_images(ema_generator, fixed_noise * NOISE_SCALE)
    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    for img, ax in zip(imgs, axes.flat):
        ax.imshow(img); ax.axis("off")
    plt.suptitle(f"Epoch {epoch} (EMA)")
    plt.tight_layout()
    plt.savefig(os.path.join(SAMPLES_PATH, f"epoch_{epoch:03d}.png"), dpi=100)
    if show:
        plt.show()
    plt.close()

history = []
for epoch in range(1, EPOCHS + 1):
    lr = LR_START + (LR_END - LR_START) * (epoch - 1) / max(1, EPOCHS - 1)   # بينزل تدريجيًا
    set_lr(g_opt, lr); set_lr(d_opt, lr)

    t0 = time.time()
    g_losses, d_losses = [], []
    for batch in tqdm(train_dataset, desc=f"Epoch {epoch}/{EPOCHS}"):
        g, d = train_step(batch)
        g_losses.append(float(g)); d_losses.append(float(d))

    history.append({"Epoch": epoch,
                    "Generator Loss": float(np.mean(g_losses)),
                    "Discriminator Loss": float(np.mean(d_losses))})
    print(f"Epoch {epoch} | G: {history[-1]['Generator Loss']:.3f} "
          f"| D: {history[-1]['Discriminator Loss']:.3f} "
          f"| lr {lr:.1e} | {(time.time() - t0) / 60:.1f} min")

    save_epoch_images(epoch, show=(epoch == 1 or epoch % 5 == 0))

    if epoch % 5 == 0:
        ema_generator.save(os.path.join(NEW_PATH, "generator_ema.keras"))
        generator.save(os.path.join(NEW_PATH, "generator_raw.keras"))

ema_generator.save(os.path.join(NEW_PATH, "generator_ema.keras"))
generator.save(os.path.join(NEW_PATH, "generator_raw.keras"))
discriminator.save(os.path.join(NEW_PATH, "discriminator.keras"))
print("Training finished!")

df = pd.DataFrame(history)
df.to_csv(os.path.join(NEW_PATH, "history.csv"), index=False)
plt.figure(figsize=(9, 4))
plt.plot(df["Epoch"], df["Generator Loss"], label="Generator")
plt.plot(df["Epoch"], df["Discriminator Loss"], label="Discriminator")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Losses (improved run)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(NEW_PATH, "losses.png"), dpi=150)
plt.show()

noise = tf.random.normal([8, LATENT_DIM]) * NOISE_SCALE
before = to_images(old_generator, noise)
after  = to_images(ema_generator, noise)
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for i in range(8):
    axes[0, i].imshow(before[i]); axes[0, i].axis("off")
    axes[1, i].imshow(after[i]);  axes[1, i].axis("off")
axes[0, 0].set_title("Before", loc="left", fontsize=14)
axes[1, 0].set_title("After (improved)", loc="left", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(NEW_PATH, "before_after.png"), dpi=150)
plt.show()

faces = to_images(ema_generator, tf.random.normal([25, LATENT_DIM]) * NOISE_SCALE)
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for img, ax in zip(faces, axes.flat):
    ax.imshow(img); ax.axis("off")
plt.suptitle("Final Generated Faces (improved)", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(NEW_PATH, "final_faces.png"), dpi=150)
plt.show()

shutil.make_archive("/kaggle/working/DCGAN_results_v2", "zip", NEW_PATH)
print("DONE", NEW_PATH)


In [ ]:

import os
import torch
from PIL import Image
from diffusers import DDIMPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)    

pipe = DDIMPipeline.from_pretrained("google/ddpm-celebahq-256").to(device)
OUT_DIR = "/kaggle/working/linkedin_faces"
os.makedirs(OUT_DIR, exist_ok=True)


NUM_BATCHES = 4        
BATCH_SIZE  = 4
STEPS       = 50     
SEED        = 7

torch_gen = torch.Generator(device=device).manual_seed(SEED)
faces = []
for b in range(NUM_BATCHES):
    out = pipe(batch_size=BATCH_SIZE, num_inference_steps=STEPS,
               eta=0.0, generator=torch_gen)
    faces.extend(out.images)
    print(f"Batch {b + 1}/{NUM_BATCHES} done")

for i, img in enumerate(faces):
    img.save(os.path.join(OUT_DIR, f"face_{i + 1:02d}.png"))


import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for img, ax in zip(faces, axes.flat):
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "grid.png"), dpi=150)
plt.show()
